# 02 — Cleaning del dataset LAPD

**Fase PACE**: Analyze  
**Obiettivo**: applicare la pulizia completa al dataset combinato e produrre 
`crimes_clean.parquet`, base di riferimento per tutti i notebook successivi.

**Input**: `data/processed/crimes_merged.parquet` (prodotto da `01_data_loading.ipynb`)  
**Output**: `data/processed/crimes_clean.parquet`

## 1. Setup e caricamento

In [6]:
import pandas as pd  # Importazione framework Pandas per la manipolazione del DataFrame
import numpy as np   # Importazione framework Numpy per gestire le operazioni con i numeri

pd.set_option('display.max_rows', None)         # Impostazione pandas per mostrare tutte le righe con i comandi di ispezione
pd.set_option('display.max_columns', None)      # Impostazione pandas per mostrare tutte le colonne con i comandi di ispezione
pd.set_option('display.max_info_columns', 200)  # Impostazione pandas per mostrare le informazioni di 200 colonne con il comando '.info()'

df = pd.read_parquet('../../data/processed/crimes_merged.parquet') # Creazione della variabile 'df' che conterrà i dati presenti nel
                                                                   # file parquet 'crimes_merged.parquet'

print(f"Dataset caricato: {df.shape}")                             # Visualizza il numero di righe e colonne del DataFrame

print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")  # Visualizza la quantità di memoria utilizzata per eseguire 
                                                                        # le operazioni precedenti

Dataset caricato: (3138031, 28)
Memoria: 2659.9 MB


## 2. Eliminazione colonne non rilevanti

In base all'ispezione iniziale (fase Plan) eliminiamo le colonne che non 
porteranno valore alle analisi:

- **`Crm Cd 2`, `Crm Cd 3`, `Crm Cd 4`** (>93% nulli): rappresentano crimini 
  secondari collegati al report principale, ma sono praticamente sempre vuote. 
  Per le nostre domande analitiche è sufficiente `Crm Cd` (crimine principale).
- **`Cross Street`** (84% nulli): informazione ridondante rispetto a `LOCATION` 
  e alle coordinate `LAT`/`LON`.

In [7]:
colonne_da_eliminare = ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street']

df = df.drop(columns=colonne_da_eliminare)

print(f"Colonne eliminate: {colonne_da_eliminare}")
print(f"Nuova shape: {df.shape}")
print(f"Colonne rimanenti: {df.shape[1]}")

Colonne eliminate: ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street']
Nuova shape: (3138031, 24)
Colonne rimanenti: 24


## 3. Conversione dei tipi di dato

### 3.1 Date (`Date Rptd` e `DATE OCC`)

Entrambe le colonne sono attualmente stringhe nel formato `MM/DD/YYYY HH:MM:SS AM/PM` 
(formato US). Le convertiamo in `datetime64` per poter eseguire operazioni temporali, 
filtri per anno/mese, calcoli di intervalli, ecc.

- `DATE OCC` = data in cui è avvenuto il crimine (la più importante per le analisi)
- `Date Rptd` = data in cui il crimine è stato denunciato/registrato (può essere 
  successiva a quella di occorrenza, e il delta è un'informazione interessante)

In [8]:
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')   # Conversione della colonna 'DATE OCC' nel formato datetime
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce') # Conversione della colonna 'Date Rptd' nel formato datetime

print("Tipi dopo la conversione:")  # Visualizza il testo indicato

print(df[['DATE OCC', 'Date Rptd']].dtypes)  # Visualizza il dtype delle colonne 'DATE OCC' e 'Date Rptd'

print(f"\nNulli DATE OCC: {df['DATE OCC'].isnull().sum()}")  # Mostra il numero di valori nulli della colonna 'DATE OCC'

print(f"Nulli Date Rptd: {df['Date Rptd'].isnull().sum()}")  # Mostra il numero di valori nulli della colonna 'Date Rptd'

print(f"\nRange DATE OCC: {df['DATE OCC'].min()} → {df['DATE OCC'].max()}")  # Visualizza i valori minimo e massimo della colonna 'DATE OCC'
print(f"Range Date Rptd: {df['Date Rptd'].min()} → {df['Date Rptd'].max()}") # Visualizza i valori minimo e massimo della colonna 'Date Rptd'

/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_4742/3653002372.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')
/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_4742/3653002372.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce')


Tipi dopo la conversione:
DATE OCC     datetime64[ns]
Date Rptd    datetime64[ns]
dtype: object

Nulli DATE OCC: 0
Nulli Date Rptd: 0

Range DATE OCC: 2010-01-01 00:00:00 → 2024-12-30 00:00:00
Range Date Rptd: 2010-01-01 00:00:00 → 2025-06-05 00:00:00


### 3.2 Orario (`TIME OCC`)

`TIME OCC` è attualmente un intero nel formato HHMM (es. `2130` = 21:30, 
`45` = 00:45, `0` = 00:00). Lo trasformiamo in qualcosa di più gestibile: 
una colonna `hour_occ` con solo l'ora intera (0-23), che è la granularità 
sufficiente per analisi tipo "fasce orarie più critiche" o "distribuzione 
oraria dei crimini".

La colonna originale `TIME OCC` viene sostituita.

In [9]:
# Estrazione dell'ora dall'intero HHMM
df['hour_occ'] = df['TIME OCC'] // 100 # Applicazione dell'operatore 'divisione intera' alla colonna 'TIME OCC' per estrarre i valori
                                       # che comporranno la colonna 'hour_occ'

# Verifica
print(f"Range hour_occ: {df['hour_occ'].min()} → {df['hour_occ'].max()}") # Operazione di controllo che visualizza i numeri minimo e massimo
                                                                          # della colonna 'hour_occ'

print(f"\nDistribuzione per ora (top 5):")                                # Visualizza la stringa indicata

print(df['hour_occ'].value_counts().sort_index().head(5))                 # Visualizza le prime 5 righe della colonna 'hour_occ'
                                                                          # ordinati in modo ascendente in base all'indice

print(f"\nValori fuori range (>23 o <0): {((df['hour_occ'] < 0) | (df['hour_occ'] > 23)).sum()}") 
# Visualizza l f-string indicata che contiene del testo e la somma dei valori della colonna 'hour_occ' minori di 0 e maggiori di 23
# per controllare che tutti i valori siano compresi nell'intervallo del massimo e minimo creato in precedenza 
                                                                                                    

Range hour_occ: 0 → 23

Distribuzione per ora (top 5):
hour_occ
0    130102
1     90502
2     76769
3     60937
4     48319
Name: count, dtype: int64

Valori fuori range (>23 o <0): 0


In [10]:
# Elimina la colonna 'TIME OCC' dal DataFrame e ricalcola la dimensione del DataFrame tramite '.shape'
df = df.drop(columns=['TIME OCC']) 
print(f"Colonna 'TIME OCC' eliminata. Shape attuale: {df.shape}")

Colonna 'TIME OCC' eliminata. Shape attuale: (3138031, 24)


### 3.3 Gestione coordinate sentinella

Nel notebook 01 abbiamo identificato 3.148 record con coordinate `(0, 0)`, 
che il LAPD usa come valore sentinella per indicare "posizione sconosciuta".

Le sostituiamo con `NaN` invece di eliminare i record: in questo modo i record 
restano disponibili per analisi non-geografiche (tipologia di crimine, vittime, 
orari) ma vengono automaticamente esclusi dalle analisi che richiedono 
le coordinate.

In [11]:
maschera_sentinella = (df['LAT'] == 0) & (df['LON'] == 0)
n_sentinelle = maschera_sentinella.sum()

df.loc[maschera_sentinella, ['LAT', 'LON']] = np.nan

print(f"Record con coordinate sentinella sostituite: {n_sentinelle}")
print(f"Nulli LAT dopo sostituzione: {df['LAT'].isnull().sum()}")
print(f"Nulli LON dopo sostituzione: {df['LON'].isnull().sum()}")
print(f"\nNuovo range LAT: {df['LAT'].min():.4f} → {df['LAT'].max():.4f}")
print(f"Nuovo range LON: {df['LON'].min():.4f} → {df['LON'].max():.4f}")

Record con coordinate sentinella sostituite: 3148
Nulli LAT dopo sostituzione: 3148
Nulli LON dopo sostituzione: 3148

Nuovo range LAT: 33.3427 → 34.7907
Nuovo range LON: -118.8279 → -117.6596


## 4. Gestione duplicati su DR_NO

`DR_NO` (Division of Records Number) è l'identificativo univoco del report 
di polizia. In teoria non dovrebbero esserci duplicati, ma nell'ispezione 
iniziale ne abbiamo trovati 57.809.

Prima di decidere come gestirli, ispezioniamo alcuni casi reali per capire 
**perché** esistono e in cosa differiscono le righe duplicate.

In [12]:
# Trova un esempio di DR_NO duplicato
dr_no_duplicati = df[df.duplicated(subset='DR_NO', keep=False)]['DR_NO'].unique()
print(f"DR_NO unici duplicati: {len(dr_no_duplicati)}")
print(f"Righe totali coinvolte: {df.duplicated(subset='DR_NO', keep=False).sum()}")

# Prendi i primi 3 DR_NO duplicati e mostrali
print("\n=== Primi 3 esempi di duplicati ===\n")
for dr_no in dr_no_duplicati[:3]:
    print(f"--- DR_NO: {dr_no} ---")
    print(df[df['DR_NO'] == dr_no])
    print()

DR_NO unici duplicati: 57809
Righe totali coinvolte: 115618

=== Primi 3 esempi di duplicati ===

--- DR_NO: 161804259 ---
             DR_NO  Date Rptd   DATE OCC  AREA  AREA NAME  Rpt Dist No  \
937560   161804259 2016-01-06 2016-01-06    18  Southeast         1805   
1234823  161804259 2016-01-06 2016-01-06    18  Southeast         1805   

         Part 1-2  Crm Cd       Crm Cd Desc Mocodes  Vict Age Vict Sex  \
937560          1     510  VEHICLE - STOLEN    None         0     None   
1234823         1     510  VEHICLE - STOLEN    None         0     None   

        Vict Descent  Premis Cd Premis Desc  Weapon Used Cd Weapon Desc  \
937560          None      101.0      STREET             NaN        None   
1234823         None      101.0      STREET             NaN        None   

        Status  Status Desc  Crm Cd 1                                LOCATION  \
937560      IC  Invest Cont     510.0  200 E  90TH                         ST   
1234823     IC  Invest Cont     510.0  200 

In [13]:
# Verifica se i duplicati su DR_NO sono anche duplicati esatti su tutta la riga
duplicati_su_dr_no = df.duplicated(subset='DR_NO', keep=False).sum()
duplicati_esatti_riga = df.duplicated(keep=False).sum()

print(f"Righe con DR_NO duplicato: {duplicati_su_dr_no}")
print(f"Righe duplicate esatte (tutte le colonne): {duplicati_esatti_riga}")

if duplicati_su_dr_no == duplicati_esatti_riga:
    print("\n✅ Tutti i duplicati su DR_NO sono duplicati esatti")
else:
    diff = duplicati_su_dr_no - duplicati_esatti_riga
    print(f"\n⚠️ Ci sono {diff} righe con DR_NO duplicato ma con differenze nelle colonne")

Righe con DR_NO duplicato: 115618
Righe duplicate esatte (tutte le colonne): 115618

✅ Tutti i duplicati su DR_NO sono duplicati esatti
